# Crash Severity Predictor & Severity-Weighted Hotspots — Fairfield County, CT
**Jan Salafia, Sean Salafia — Machine Learning final project**

Data: Connecticut Crash Data Repository (CTCDR), PR-1 era (2015 onward).

**Research question:** Which Fairfield County locations have the highest crash *severity*, and does a severity-weighted
hotspot ranking differ from a ranking by crash count? What conditions make crashes severe?

**Notebook outline**
1. Setup & configuration
2. Load data
3. Preprocessing (Fairfield County, 2015+), severity score and target
4. Exploratory data analysis
5. Severity predictor: baseline vs. logistic regression vs. random forest
6. Hotspots: DBSCAN clusters, ranked by severity score vs. crash count
7. What do the most severe hotspots have in common?
8. Summary & limitations

> Items marked **`# CHECK`** are things to verify against your actual CTCDR export.

## 1. Setup & configuration

In [3]:
import glob, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sklearn.cluster import DBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50, "display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
RNG = 42

# ---------------- CONFIG ----------------
DATA_GLOB = "data/*.csv"          # CHECK: put your CTCDR CSV export(s) in ./data/ (several files are concatenated)
ALLOW_SYNTHETIC = False            # if no CSV is found, generate FAKE demo data so the notebook can be tested
START_YEAR = 2015

FAIRFIELD_COUNTY_TOWNS = [
    "Bethel","Bridgeport","Brookfield","Danbury","Darien","Easton","Fairfield","Greenwich","Monroe",
    "New Canaan","New Fairfield","Newtown","Norwalk","Redding","Ridgefield","Shelton","Sherman",
    "Stamford","Stratford","Trumbull","Weston","Westport","Wilton"]

# Simple severity score (KABCO): higher = more severe. Severe = K (fatal) or A (suspected serious injury).
SEVERITY_SCORE = {"K": 5, "A": 4, "B": 3, "C": 2, "O": 1}

# CHECK: our short name -> column name in your CTCDR export. Use either "datetime" or "date" (+ optional "time").
COLS = {
    "crash_id": "Crash ID",
    "datetime": "Crash Date Time",
    "town":     "Town Name",
    "lat":      "Latitude",
    "lon":      "Longitude",
    "severity": "Crash Severity",
}

# CHECK: model features (column names in the export). Never include anything only known after the outcome
# (e.g. number injured, ambulance/transport) — that would leak the answer.
CAT_FEATURES = ["Manner of Crash/Collision Impact", "Light Condition", "Weather Condition",
                "Road Surface Condition", "Route Class", "Contributing Factor"]
NUM_FEATURES = ["Posted Speed Limit", "Number of Vehicles"]
TIME_FEATURES = ["hour", "dow", "month"]   # created from the crash date/time below

# Hotspot clustering
EPS_METERS = 100        # crashes within this distance chain into one hotspot
MIN_CRASHES = 15        # minimum crashes to call a cluster a hotspot
TOP_N = 25              # size of the "top list" used when comparing rankings

## 2. Load data

In [2]:
def make_synthetic(n=60000, seed=0):
    """FAKE data with CTCDR-like columns. Only for testing the pipeline — results are meaningless."""
    rng = np.random.default_rng(seed)
    towns = {"Bridgeport":(41.1792,-73.1894,9),"Stamford":(41.0534,-73.5387,9),"Norwalk":(41.1177,-73.4082,7),
             "Danbury":(41.3948,-73.4540,6),"Fairfield":(41.1408,-73.2613,5),"Greenwich":(41.0262,-73.6282,4),
             "Stratford":(41.1845,-73.1332,4),"Shelton":(41.3165,-73.0932,3),"Trumbull":(41.2429,-73.2007,3),
             "Westport":(41.1415,-73.3579,3),"Bethel":(41.3715,-73.4140,2),"Brookfield":(41.4826,-73.4098,2),
             "Darien":(41.0787,-73.4693,2),"Easton":(41.2534,-73.2989,1),"Monroe":(41.3326,-73.2073,2),
             "New Canaan":(41.1465,-73.4948,2),"New Fairfield":(41.4676,-73.4890,1),"Newtown":(41.4140,-73.3040,2),
             "Redding":(41.3025,-73.3837,1),"Ridgefield":(41.2815,-73.4982,2),"Sherman":(41.5818,-73.4970,1),
             "Weston":(41.2007,-73.3812,1),"Wilton":(41.1954,-73.4379,2)}
    names = list(towns); w = np.array([towns[t][2] for t in names], float); w /= w.sum()
    town = rng.choice(names, n, p=w)
    base = np.array([towns[t][:2] for t in town])
    # 40% of crashes sit on ~60 "problem spots", the rest scatter around town centers
    spots = base[rng.choice(n, 60, replace=False)] + rng.normal(0, .004, (60, 2))
    spot_risk = rng.uniform(0.6, 2.2, 60)
    on_spot = rng.random(n) < 0.4
    sid = rng.integers(0, 60, n)
    ll = np.where(on_spot[:, None], spots[sid] + rng.normal(0, .0004, (n, 2)), base + rng.normal(0, .02, (n, 2)))
    risk = np.where(on_spot, spot_risk[sid], 1.0)
    dt = pd.to_datetime("2015-01-01") + pd.to_timedelta(rng.integers(0, 365*10 + 330, n), unit="D") \
         + pd.to_timedelta(rng.integers(0, 24, n), unit="h") \
         + pd.to_timedelta(rng.integers(0, 60, n), unit="m")
    light = rng.choice(["Daylight","Dark - Lighted","Dark - Not Lighted","Dusk","Dawn"], n, p=[.65,.18,.08,.06,.03])
    weather = rng.choice(["Clear","Rain","Snow","Cloudy","Fog"], n, p=[.6,.15,.05,.18,.02])
    surf = np.where(weather=="Snow", "Snow/Ice", np.where(weather=="Rain", "Wet", "Dry"))
    coll = rng.choice(["Rear-end","Angle","Sideswipe","Fixed object","Head-on","Pedestrian","Backing"], n,
                      p=[.30,.20,.12,.15,.03,.03,.17])
    route = rng.choice(["Local road","State route","Interstate","US route"], n, p=[.5,.25,.15,.10])
    speed = rng.choice([25,30,35,40,45,55,65], n, p=[.15,.25,.2,.15,.1,.1,.05])
    nveh = rng.choice([1,2,3,4], n, p=[.3,.55,.1,.05])
    factor = rng.choice(["None","Following too closely","Failed to grant right of way","Speeding","Distracted","Impaired","Unsafe lane change"],
                        n, p=[.2,.22,.18,.1,.15,.05,.10])
    z = (-4.3 + 0.015*(speed-35) + 0.6*(light=="Dark - Not Lighted") + 0.2*(light=="Dark - Lighted")
         + 1.3*(coll=="Pedestrian") + 1.0*(coll=="Head-on") + 0.5*(coll=="Fixed object") + 0.4*(coll=="Angle")
         - 0.6*(coll=="Backing") - 0.3*(coll=="Rear-end") + 0.5*(factor=="Impaired") + 0.4*(factor=="Speeding")
         + 0.3*(route=="Interstate") + np.log(risk))
    p = 1/(1+np.exp(-z))
    u = rng.random(n)
    sev = np.where(u < p*0.12, "K - Fatal Injury", np.where(u < p, "A - Suspected Serious Injury",
          np.where(u < p + .06, "B - Suspected Minor Injury", np.where(u < p + .15, "C - Possible Injury", "O - No Apparent Injury"))))
    df = pd.DataFrame({"Crash ID": np.arange(n), "Crash Date Time": dt.strftime("%m/%d/%Y %I:%M:%S %p"),
        "Town Name": town, "Latitude": ll[:, 0], "Longitude": ll[:, 1], "Crash Severity": sev,
        "Manner of Crash/Collision Impact": coll, "Light Condition": light, "Weather Condition": weather,
        "Road Surface Condition": surf, "Route Class": route, "Posted Speed Limit": speed,
        "Number of Vehicles": nveh, "Contributing Factor": factor,
        "Number of Injured": np.where(np.isin(sev, ["O - No Apparent Injury"]), 0, rng.integers(1, 3, n))})
    return df

files = sorted(glob.glob(DATA_GLOB))
USING_SYNTHETIC = False
if files:
    raw = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
    print(f"Loaded {len(files)} file(s), {len(raw):,} rows")
elif ALLOW_SYNTHETIC:
    raw = make_synthetic(); USING_SYNTHETIC = True
    print("!!! NO CSV FOUND — using FAKE synthetic data. Do not interpret any result. !!!")
else:
    raise FileNotFoundError(f"No files match {DATA_GLOB}")

print(raw.shape)
missing = [c for c in list(COLS.values()) + CAT_FEATURES + NUM_FEATURES if c not in raw.columns]
print("Columns not found (edit COLS / features above):", missing)
raw.head(3)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x93 in position 62525: invalid start byte

## 3. Preprocessing
- Rename the key columns, remove duplicate crash records, report missing values
- Parse the crash date/time; keep 2015 onward and Fairfield County towns
- Convert severity text to KABCO letters → severity score (K=5 … O=1) and binary target `severe` (K or A)
- Coordinates outside the county are set to missing (those crashes stay in the model but not in the hotspots)
- Missing categorical values become `"Missing"`, missing numbers get the median (done inside the model pipeline)

In [ ]:
def parse_kabco(s):
    """Map CTCDR severity text to K/A/B/C/O. Handles 'K - Fatal Injury', 'K', or plain descriptions."""
    if pd.isna(s): return np.nan
    t = str(s).strip().upper()
    m = re.match(r"^([KABCO])\b", t)
    if m: return m.group(1)
    if "FATAL" in t: return "K"
    if "SERIOUS" in t: return "A"
    if "MINOR" in t: return "B"
    if "POSSIBLE" in t: return "C"
    if "NO APPARENT" in t or "NO INJURY" in t or "PROPERTY" in t: return "O"
    return np.nan

df = raw.rename(columns={v: k for k, v in COLS.items()})
print(f"Duplicate crash IDs removed: {df.crash_id.duplicated().sum():,}")
df = df.drop_duplicates(subset="crash_id")

print("\nMissing values (% of rows):")
print((df[list(COLS) + [c for c in CAT_FEATURES + NUM_FEATURES if c in df]].isna().mean() * 100).round(1))

# Date / time
if "datetime" in df:
    df["dt"] = pd.to_datetime(df["datetime"], errors="coerce")
elif "time" in df:
    df["dt"] = pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str), errors="coerce")
else:
    df["dt"] = pd.to_datetime(df["date"], errors="coerce")

df["town"] = df.town.astype(str).str.strip().str.title()
df["kabco"] = df.severity.map(parse_kabco)
print("\nUnparsed severity values:", df.loc[df.kabco.isna(), "severity"].value_counts(dropna=False).head(8).to_dict())

steps = {"after duplicates": len(df)}
df = df[df.dt.notna()];                         steps["valid date"] = len(df)
df = df[df.dt.dt.year >= START_YEAR];           steps[f">= {START_YEAR}"] = len(df)
df = df[df.town.isin(FAIRFIELD_COUNTY_TOWNS)];  steps["Fairfield County"] = len(df)
df = df[df.kabco.notna()];                      steps["severity known"] = len(df)
print(pd.Series(steps).to_frame("rows remaining"))

In [ ]:
# Coordinates: keep only points inside a Fairfield County bounding box
df["lat"] = pd.to_numeric(df.lat, errors="coerce")
df["lon"] = pd.to_numeric(df.lon, errors="coerce")
bad_xy = ~(df.lat.between(40.9, 41.7) & df.lon.between(-73.8, -73.0))
df.loc[bad_xy, ["lat", "lon"]] = np.nan
print(f"Missing/invalid coordinates: {df.lat.isna().mean():.1%}")

# Target and severity score
df["sev_score"] = df.kabco.map(SEVERITY_SCORE)
df["severe"] = df.kabco.isin(["K", "A"]).astype(int)

# Time features
df["year"] = df.dt.dt.year
df["hour"] = df.dt.dt.hour
df["dow"] = df.dt.dt.dayofweek      # 0 = Monday
df["month"] = df.dt.dt.month

# Categorical features: consistent text, missing -> "Missing"
for c in CAT_FEATURES:
    df[c] = df[c].fillna("Missing").astype(str).str.strip()

df = df.reset_index(drop=True)
print(f"Final rows: {len(df):,}  |  years {df.year.min()}–{df.year.max()}  |  severe (K+A) share: {df.severe.mean():.2%}")

## 4. Exploratory data analysis

In [ ]:
counts = df.kabco.value_counts().reindex(list("KABCO"))
print(counts.to_frame("crashes").assign(share=(counts / counts.sum()).round(4)))

ax = counts.plot(kind="bar", color="#2a6f97", rot=0)
ax.set(title="Crashes by severity (KABCO)", xlabel="", ylabel="Crashes"); plt.show()

Severe crashes are only a few percent of all crashes, so this is an **imbalanced** problem: accuracy is misleading, and we use class weights plus recall, F1 and PR-AUC.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for j, (col, ttl) in enumerate([("hour", "hour of day"), ("dow", "day of week (0=Mon)"), ("year", "year")]):
    g = df.groupby(col).severe.agg(["size", "mean"])
    axes[0, j].bar(g.index, g["size"], color="#2a6f97"); axes[0, j].set_title(f"Crashes by {ttl}")
    axes[1, j].bar(g.index, g["mean"] * 100, color="#b23a48"); axes[1, j].set_title(f"% severe by {ttl}")
plt.tight_layout(); plt.show()

In [ ]:
# Severe-crash rate by condition (categories with at least 200 crashes)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), CAT_FEATURES):
    g = df.groupby(col).severe.agg(["size", "mean"])
    g = g[g["size"] >= 200].sort_values("mean")
    ax.barh(g.index, g["mean"] * 100, color="#b23a48")
    ax.axvline(df.severe.mean() * 100, ls="--", c="grey", lw=1)
    ax.set_title(col); ax.set_xlabel("% severe (dashed = overall)")
plt.tight_layout(); plt.show()

In [ ]:
geo = df.dropna(subset=["lat", "lon"]).copy()
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(geo.lon, geo.lat, s=1, c="lightgrey", label="All crashes")
s = geo[geo.severe == 1]
ax.scatter(s.lon, s.lat, s=5, c="#b23a48", label="Severe (K+A)")
ax.set(title="Crash locations, Fairfield County", xlabel="Longitude", ylabel="Latitude"); ax.legend(); plt.show()

## 5. Severity predictor
**Target:** `severe` = fatal (K) or suspected serious injury (A). **Features:** conditions recorded on the crash report plus hour, day of week and month.

**Split by time:** train on earlier years, test on the most recent two years (mimics real use: learn from the past, predict the future).

**Models:** a baseline that always predicts "not severe", logistic regression, and random forest — both with
`class_weight="balanced"` so the rare severe class isn't ignored.

In [ ]:
test_years = [int(y) for y in sorted(df.year.unique())[-2:]]
train, test = df[~df.year.isin(test_years)], df[df.year.isin(test_years)]
FEATS = CAT_FEATURES + NUM_FEATURES + TIME_FEATURES
Xtr, ytr, Xte, yte = train[FEATS], train.severe, test[FEATS], test.severe
print(f"train {len(train):,} (before {test_years[0]}) | test {len(test):,} {test_years}")
print(f"severe rate  train {ytr.mean():.2%} | test {yte.mean():.2%}")

prep = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), CAT_FEATURES),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM_FEATURES + TIME_FEATURES)])

models = {
    "Baseline (always not severe)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                            n_jobs=-1, random_state=RNG),
}
fitted, rows = {}, []
for name, model in models.items():
    pipe = Pipeline([("prep", prep), ("clf", model)]).fit(Xtr, ytr)
    fitted[name] = pipe
    pred, prob = pipe.predict(Xte), pipe.predict_proba(Xte)[:, 1]
    rows.append({"model": name, "precision": precision_score(yte, pred, zero_division=0),
                 "recall": recall_score(yte, pred), "F1": f1_score(yte, pred),
                 "ROC-AUC": roc_auc_score(yte, prob), "PR-AUC": average_precision_score(yte, prob)})
res = pd.DataFrame(rows).set_index("model")
print(f"\nPR-AUC of a random guess ≈ test severe rate = {yte.mean():.3f}")
res.round(3)

In [ ]:
for name in ["Logistic regression", "Random forest"]:
    print(f"===== {name} =====")
    print(classification_report(yte, fitted[name].predict(Xte), target_names=["Not severe", "Severe"]))

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, name in zip(axes[:2], ["Logistic regression", "Random forest"]):
    ConfusionMatrixDisplay.from_estimator(fitted[name], Xte, yte, display_labels=["Not severe", "Severe"],
                                          ax=ax, colorbar=False)
    ax.set_title(name)
for name in ["Logistic regression", "Random forest"]:
    PrecisionRecallDisplay.from_estimator(fitted[name], Xte, yte, name=name, ax=axes[2])
    RocCurveDisplay.from_estimator(fitted[name], Xte, yte, name=name, ax=axes[3])
axes[2].axhline(yte.mean(), ls="--", c="grey", lw=1); axes[2].set_title("Precision–Recall (test)")
axes[3].set_title("ROC (test)")
plt.tight_layout(); plt.show()

### Which features matter?

In [ ]:
names = fitted["Random forest"].named_steps["prep"].get_feature_names_out()
names = [n.split("__", 1)[1] for n in names]
rf_imp = pd.Series(fitted["Random forest"].named_steps["clf"].feature_importances_, index=names)
lr_coef = pd.Series(fitted["Logistic regression"].named_steps["clf"].coef_[0], index=names)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
rf_imp.sort_values().tail(15).plot(kind="barh", ax=axes[0], color="#2a6f97")
axes[0].set_title("Random forest: top 15 feature importances")
top = lr_coef.reindex(lr_coef.abs().sort_values().tail(15).index)
top.plot(kind="barh", ax=axes[1], color=np.where(top > 0, "#b23a48", "#2a6f97"))
axes[1].set_title("Logistic regression: largest coefficients\n(red = raises odds of a severe crash)")
plt.tight_layout(); plt.show()

## 6. Hotspots: cluster crashes, then rank by severity vs. by count
Crash locations are converted to metres and grouped with **DBSCAN**: crashes within `EPS_METERS` of each other chain into
one cluster, and a cluster needs at least `MIN_CRASHES` crashes to count as a hotspot (isolated crashes are noise).
Each hotspot is then ranked two ways:
- **by crash count** — the standard approach
- **by total severity score** (K=5 … O=1) — our approach, where severe crashes count more

In [ ]:
lat0 = geo.lat.mean()
XY = np.c_[geo.lon * 111_320 * np.cos(np.radians(lat0)), geo.lat * 110_540]   # approx. metres, fine at county scale
geo["hotspot"] = DBSCAN(eps=EPS_METERS, min_samples=MIN_CRASHES, n_jobs=-1).fit_predict(XY)

hot = geo[geo.hotspot >= 0].groupby("hotspot").agg(
    town=("town", lambda s: s.mode().iat[0]), lat=("lat", "mean"), lon=("lon", "mean"),
    crashes=("severe", "size"), severe=("severe", "sum"), fatal=("kabco", lambda s: (s == "K").sum()),
    severity_score=("sev_score", "sum")).reset_index()
hot["severe_rate"] = (hot.severe / hot.crashes).round(3)
hot["rank_count"] = hot.crashes.rank(ascending=False, method="min").astype(int)
hot["rank_severity"] = hot.severity_score.rank(ascending=False, method="min").astype(int)
hot = hot.sort_values("rank_severity")

print(f"{len(hot)} hotspots contain {(geo.hotspot >= 0).mean():.1%} of geocoded crashes")
hot.head(15).round({"lat": 4, "lon": 4})

In [ ]:
rho = spearmanr(hot.rank_count, hot.rank_severity)[0]
n = min(TOP_N, len(hot))
top_count = set(hot.nsmallest(n, "rank_count").hotspot)
top_sev = set(hot.nsmallest(n, "rank_severity").hotspot)
print(f"Spearman correlation between count rank and severity rank: {rho:.3f}")
print(f"Top-{n} lists share {len(top_count & top_sev)} hotspots ({n - len(top_count & top_sev)} differ)")

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
hidden = hot[hot.hotspot.isin(top_sev - top_count)]
ax[0].scatter(hot.crashes, hot.severity_score, s=20, c="#2a6f97", label="hotspot")
ax[0].scatter(hidden.crashes, hidden.severity_score, s=50, c="#b23a48", label=f"top-{n} by severity, not by count")
ax[0].set(xlabel="Crash count", ylabel="Total severity score", title="Count vs. severity per hotspot"); ax[0].legend()

ax[1].scatter(geo.lon, geo.lat, s=1, c="lightgrey")
tc, ts = hot[hot.hotspot.isin(top_count)], hot[hot.hotspot.isin(top_sev)]
ax[1].scatter(tc.lon, tc.lat, s=120, facecolors="none", edgecolors="#2a6f97", linewidths=2, label=f"Top {n} by count")
ax[1].scatter(ts.lon, ts.lat, s=40, c="#b23a48", marker="x", label=f"Top {n} by severity")
ax[1].set(xlabel="Longitude", ylabel="Latitude", title="Top hotspots: count vs. severity ranking"); ax[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

print(f"'Hidden' hotspots: top-{n} by SEVERITY but not top-{n} by COUNT (what a count-only ranking misses)")
hidden[["hotspot", "town", "lat", "lon", "crashes", "severe", "fatal", "severity_score", "rank_count", "rank_severity"]].round(4)

## 7. What do the most severe hotspots have in common?
Share of each condition among crashes in the top hotspots (by severity) vs. all other crashes.
**Lift** > 1 means the condition is over-represented in the most severe hotspots.

In [ ]:
geo["in_top"] = geo.hotspot.isin(top_sev)
rows = []
for c in CAT_FEATURES:
    a = geo.loc[geo.in_top, c].value_counts(normalize=True)
    b = geo.loc[~geo.in_top, c].value_counts(normalize=True)
    n_top = geo.loc[geo.in_top, c].value_counts()
    for v in a.index:
        if n_top[v] >= 30:
            rows.append({"feature": c, "value": v, "share_in_top": a[v], "share_elsewhere": b.get(v, 0.0),
                         "lift": a[v] / max(b.get(v, 0.0), 1e-9), "n_top": n_top[v]})
lift = pd.DataFrame(rows).sort_values("lift", ascending=False)
display(lift.head(12).round(3))

ax = (geo[geo.in_top].hour.value_counts(normalize=True).sort_index() * 100).plot(label="Top severe hotspots", color="#b23a48")
(geo[~geo.in_top].hour.value_counts(normalize=True).sort_index() * 100).plot(ax=ax, label="Elsewhere", color="#2a6f97")
ax.set(title="Hour of day", xlabel="Hour", ylabel="% of crashes"); ax.legend(); plt.show()

os.makedirs("outputs", exist_ok=True)
hot.to_csv("outputs/hotspots_ranked.csv", index=False)
if USING_SYNTHETIC: print("\n*** REMINDER: every number above came from FAKE data. ***")

## 8. Summary & limitations
*(Fill in after running on the real CTCDR data.)*

- **Prediction:** how much better the models do than the baseline (recall, F1, PR-AUC vs. the severe-crash rate) and which conditions matter most.
- **Hotspots:** how much the top-25 lists by crash count and by severity overlap; the "hidden" hotspots are where a count-only ranking would under-prioritize.
- **Common characteristics:** conditions over-represented at the most severe hotspots (lighting, collision type, road type, time of day).

**Limitations**
- The severity score (K=5 … O=1) is a simple ordinal weight, not a dollar cost; the FHWA Crash Cost tool is future work.
- No traffic volume (AADT) in the crash data, so a high count may just mean a busy road.
- The model predicts severity *given a crash happened*; feature importance shows association, not causation.
- DBSCAN results depend on `EPS_METERS` and `MIN_CRASHES`; long busy corridors (e.g. I-95) can chain into one large hotspot.
- Severe crashes are rare, so precision is low and results for small categories are noisy.